# 6. Combining experiments, and how firm a result is

Two things that matter once a search produces numbers someone might act on: **where two
experiments can share ground**, and **how much the answer depends on assumptions**.

The second is the more important, and it is the one most easily skipped.

In [1]:
import os
import sys

sys.path.insert(0, os.path.abspath(os.path.join('..', 'src')))

import numpy as np

import combine_experiments as combine
import physics

## Combining is an overlay, and alignment is not optional

Each experiment is one run of the searcher with its own configuration, so combining them
is an overlay of the masks those runs produce. Three questions get different answers:

- **joint** — terrain satisfying *every* experiment. One site, one road, one power feed,
  two experiments.
- **union** — terrain satisfying *any*. How much of the region is useful to the
  programme as a whole.
- **each alone** — and what each would lose by being confined to the joint area.

The inputs must be pixel-aligned: same shape, same pixel size, same corner. That is
checked and **refused** rather than resampled, because two runs on differently-cropped
DEMs would silently compare the wrong ground.

In [2]:
# What the check actually compares: the six affine terms of the world file
world_a = (1/3600, 0.0, 0.0, -1/3600, -72.4, -15.3)
world_b = (1/3600, 0.0, 0.0, -1/3600, -72.1, -15.3)      # shifted 0.3 deg east

runs = [{"dir": "run_a", "mask": np.zeros((10, 10), bool), "world": world_a},
        {"dir": "run_b", "mask": np.zeros((10, 10), bool), "world": world_b}]

try:
    combine.check_alignment(runs)
except SystemExit as exc:
    print("refused, correctly:\n")
    print(exc)

refused, correctly:

masks do not cover the same ground: upper-left x is -72.4 in run_a and -72.1 in run_b.


Same shape, same pixel size, different ground — and it says so rather than overlaying
them.

## Reading a co-location result

On the Colca crop, with both experiments run over identical terrain:

| | area | sites | capacity | of its own area in the joint |
|---|---|---|---|---|
| GRAND | 4580.2 km² | 1 | 5317 | 1.2% |
| TAMBO | 93.1 km² | 17 | 10 878 | 58.9% |
| **joint** | 54.9 km² | | | Jaccard 0.012 |

The interesting part is *why* the joint is small. Two thirds of TAMBO-viable ground is
also GRAND-viable, but the two deployable **slope bands barely overlap** — GRAND's 3–25°
against Colca's ~40° walls leaves only a 20–25° sliver. Co-location is decided by slope,
not by arrival geometry.

In [3]:
grand = (3.0, 25.0)
tambo = (20.0, 60.0)
lo, hi = max(grand[0], tambo[0]), min(grand[1], tambo[1])
print(f"GRAND deployable band: {grand[0]:.0f}-{grand[1]:.0f} deg")
print(f"TAMBO wall band:       {tambo[0]:.0f}-{tambo[1]:.0f} deg")
print(f"overlap:               {lo:.0f}-{hi:.0f} deg  ({hi-lo:.0f} deg wide)")

GRAND deployable band: 3-25 deg
TAMBO wall band:       20-60 deg
overlap:               20-25 deg  (5 deg wide)


## How firm is a result?

`oroscope-sensitivity` varies one parameter at a time about a baseline and tabulates how
much each moves the answer. Run against a real TAMBO baseline, the verdict was blunt:

| parameter | | | | |
|---|---|---|---|---|
| `decay_energy_pev` | 3 → **10 878** | 55 → **2056** | 100 → **0** | 1000 → **0** |
| `min_score` | 0.0 → **45 928** | 0.2 → **15 481** | 0.35 → **2056** | 0.5 → **0** |
| `min_target_slope_deg` | 0° → **7442** | 15° → **5309** | 25° → **2056** | 35° → **0** |

Every criterion sits near a cliff. **The decay energy is the worst**: across TAMBO's own
3 PeV – 1 EeV reach the answer runs from 10 878 to zero, because a single energy cannot
stand in for a spectrum.

In [4]:
crossing_m = 3000.0
print(f"P(tau decays within a {crossing_m/1000:.0f} km crossing):\n")
for e in (3.0, 10.0, 55.0, 100.0, 1000.0):
    L = physics.tau_decay_length_m(e)
    p = 1 - np.exp(-crossing_m / L)
    bar = "#" * int(round(p * 40))
    print(f"{e:>7.0f} PeV  {p:5.3f}  {bar}")

P(tau decays within a 3 km crossing):

      3 PeV  1.000  ########################################
     10 PeV  0.998  ########################################
     55 PeV  0.672  ###########################
    100 PeV  0.458  ##################
   1000 PeV  0.059  ##


That is a factor of seventeen inside one experiment's energy reach — and it is invisible
to every other term in the score.

**So: fold over the real spectrum before quoting a capacity.** A number computed at one
representative energy is a property of the energy chosen, not of the terrain.

## What to distrust in your own results

Three things this project measured about itself, worth checking in any search:

1. **Reported area is not physics-accepted area.** Morphological closing more than
   doubles it — measured at 2.29× with a stride-1 control run. Closing is not wrong; a
   site has to be a deployable region rather than a scatter of pixels. But the two
   numbers are different and should not be conflated.
2. **Candidate striding is unbiased** — acceptance is identical at strides 1 and 5, and
   the stride-corrected area matches the stride-1 truth to 0.05%. So that one *is* safe.
3. **Area and capacity are measured on different grids** at `downsample_factor > 1`, so
   a feature a few pixels wide loses area it keeps detectors on.

`docs/assumptions.rst` is the full list, and it is deliberately blunt.

## Where to go next

- The **[assumptions and limitations](https://mbustama.github.io/oroscope/assumptions.html)**
  page — what the numbers rest on.
- The **[physics](https://mbustama.github.io/oroscope/physics.html)** page — the
  derivation behind every criterion.

---

*Part of the [Oroscope](https://github.com/mbustama/oroscope) tutorials. Previous: [GRAND and TAMBO](05_grand_and_tambo.ipynb). Full API reference: [oroscope docs](https://mbustama.github.io/oroscope/functions.html).*